# Magnetic Resonance Imaging II

## Exercise 1 - 2D Gradient-echo sequence with simulation via MRZero and reconstruction

**Prof. Dr. Markus Zimmermann**

**Mohamed Wassim Bensalem**

## Software Imports

Hier werden alle benötigten Bibliotheken und Module importiert.

In [ ]:
%matplotlib widget

import os
import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

## Parameters

Description here

In [ ]:
plot = True
test_report = False
write_seq = True
seq_filename = "gre.seq"
fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
flip_angle_deg = 90
slice_thickness = 3e-3
# tr = 12e-3
tr = 5000e-3
te = 100e-3
# te = 5e-3
rf_spoiling_inc = 117

torch.set_default_device("cpu")

## System

Description here

In [ ]:
# Set system limits
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
    rf_ringdown_time=20e-6,
    rf_dead_time=100e-6,
    adc_dead_time=10e-6,
)

seq = pp.Sequence(system)

## RF-Pulse and Slice-Selection Gradients

Description here

In [ ]:
# Your solution here

rf, gz, _ = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    duration=3e-3,
    slice_thickness=slice_thickness,
    apodization=0.42,
    time_bw_product=4,
    system=system,
    return_gz=True,
    delay=system.rf_dead_time,
    use="excitation",
)

gz_reph = pp.make_trapezoid(
    channel="z",
    area=-gz.area / 2,
    duration=1e-3,
    system=system,
)

## Frequency- and Phase-Encoding Gradients

Description here

In [ ]:
# Your solution here

delta_kx = 1 / fov_x
delta_ky = 1 / fov_y

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    duration=1e-3,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

## Spoiler Gradients

Description here

In [ ]:
# Your solution here

gx_spoil = pp.make_trapezoid(
    channel="x",
    area=2 * n_x * delta_kx,
    system=system,
)

gy_spoil = pp.make_trapezoid(
    channel="y",
    area=2 * n_y * delta_ky,
    system=system,
)

gz_spoil = pp.make_trapezoid(
    channel="z",
    area=4 / slice_thickness,
    system=system,
)

## Timing

Description here

In [ ]:
# Your solution here

EPS = 1e-12

# Calculate timing
te_delay = (
    te
    - (pp.calc_duration(gz, rf) - pp.calc_rf_center(rf)[0] - rf.delay)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
    - EPS
)
te_delay = np.ceil(te_delay / seq.grad_raster_time) * seq.grad_raster_time

tr_delay = (
    tr
    - pp.calc_duration(gz, rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - te_delay
)
tr_delay = np.ceil(tr_delay / seq.grad_raster_time) * seq.grad_raster_time

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= pp.calc_duration(gx_spoil, gz_spoil))

## TO BE NAMED

Description here

In [ ]:
# Your solution here

phase_areas = np.linspace(-n_y / 2, n_y / 2, num=n_y, endpoint=False) * delta_ky

rf_phase = 0
rf_inc = 0

# Loop over phase encodes and define sequence blocks
for phase_area in phase_areas:

    rf.phase_offset = rf_phase / 180 * np.pi
    adc.phase_offset = rf_phase / 180 * np.pi

    rf_inc = divmod(rf_inc + rf_spoiling_inc, 360.0)[1]
    rf_phase = divmod(rf_phase + rf_inc, 360.0)[1]

    seq.add_block(rf, gz)
    # seq.add_block(rf)

    gy_pre = pp.make_trapezoid(
        channel="y",
        area=phase_area,
        duration=pp.calc_duration(gx_pre),
        system=system,
    )

    seq.add_block(gx_pre, gy_pre, gz_reph)
    # seq.add_block(gx_pre, gy_pre)

    seq.add_block(pp.make_delay(te_delay))

    seq.add_block(gx, adc)

    # gy_pre.amplitude = -gy_pre.amplitude

    seq.add_block(pp.make_delay(tr_delay), gx_spoil, gy_spoil, gz_spoil)
    # seq.add_block(pp.make_delay(tr_delay), gx_spoil, gy_spoil)

## Check

Description here

In [ ]:
# Your solution here

ok, error_report = seq.check_timing()
if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

if test_report:
    print(seq.test_report())

## Plot

Description here

In [ ]:
# Your solution here

if plot:
    seq.plot(time_range=(0.0, tr))

## Write sequence

Description here

In [ ]:
# Your solution here

seq.set_definition("FOV", [fov_x, fov_y, slice_thickness])
seq.set_definition("Name", "gre")

# if write_seq:
#     seq.write("gre.seq")

In [ ]:
# phantom = mr0.util.load_phantom(size=(64, 64))
phantom = mr0.util.load_phantom(size=(n_x, n_y))
phantom.plot()

In [ ]:
data = phantom.build()

## Simulation

Description here

In [ ]:
seq.write("tmp.seq")
seq = mr0.Sequence.import_file("tmp.seq")
os.remove("tmp.seq")

In [ ]:
graph = mr0.compute_graph(seq, data, 200, 0.0001)
signal = mr0.execute_graph(graph, seq, data, 0.01, 0.01, print_progress=True)

## Reconstruction

Description here

In [ ]:
# Your solution here

signal = signal.numpy()
signal = signal.reshape((n_x, n_y))

image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(signal)))

## Display results

Description here

In [ ]:
# Your solution here

plt.figure()
plt.imshow(np.abs(image), cmap="gray")
plt.title("Reconstructed GRE Image")
plt.show()